# Setup & Data Preparation

This notebook covers the initial setup of the **vembed-factory** environment and data preparation.
Run this notebook first before proceeding to training or distillation tasks.

## Installation

In [1]:
# Clone the repository (if not running locally)
import os

# If running in Colab/Kaggle, uncomment below:
# !git clone https://github.com/fangzhensheng/vembed-factory.git
# os.chdir("vembed-factory")

print(f"Current working directory: {os.getcwd()}")

In [ ]:
# Install dependencies (using uv is recommended, but pip works too)
# !pip install -e .
!pip install -U pip
!pip install -r requirements.txt

## Data Preparation
We will prepare the Flickr30k dataset (or dummy data if Flickr30k is not available).

**Note:** For real training, you should download the Flickr30k images and place them in `data/flickr30k/images`.

In [ ]:
# This script generates dummy jsonl files if real data is missing
!python examples/prepare_data.py flickr30k

In [ ]:
# Verify Data
import os

if os.path.exists("data/flickr30k/train.jsonl"):
    print("Flickr30k data ready")
    print("Train:", "data/flickr30k/train.jsonl")
else:
    print("Using dummy data")
    print("Train:", "data/dummy/train.jsonl")

# Training CLIP with MRL

This section demonstrates how to fine-tune a CLIP model using **Matryoshka Representation Learning (MRL)**.

In [ ]:
import os

# Ensure we are in the project root
if os.path.exists("vembed-factory"):
    os.chdir("vembed-factory")
elif os.getcwd().endswith("notebooks"):
    os.chdir("..")

print(f"Working Directory: {os.getcwd()}")

In [ ]:
# Config Data Paths
if os.path.exists("data/flickr30k/train.jsonl"):
    DATA_PATH = "data/flickr30k/train.jsonl"
    IMAGE_ROOT = "data/flickr30k"
    VAL_DATA_PATH = "data/flickr30k/val.jsonl"
else:
    DATA_PATH = "data/dummy/train.jsonl"
    IMAGE_ROOT = "data/dummy"
    VAL_DATA_PATH = ""

## Start Training
We use the standardized YAML configuration from `examples/clip_train.yaml`.
- Uses `clip_train.yaml` with MRL enabled
- Override specific parameters as needed

In [ ]:
!python run.py examples/clip_train.yaml \
    --data_path $DATA_PATH \
    --val_data_path "$VAL_DATA_PATH" \
    --image_root "$IMAGE_ROOT" \
    --config_override \
        output_dir=experiments/output_clip_mrl \
        epochs=3 \
        batch_size=32 \
        use_mrl=true \
        mrl_dims=[512,256,128]

## Evaluation
We use the benchmark script to evaluate the MRL-trained model.

In [ ]:
if os.path.exists(VAL_DATA_PATH):
    # Evaluate on the validation set using MRL
    !python benchmark/run.py flickr30k \
        --model_path experiments/output_clip_mrl/checkpoint-epoch-3 \
        --flickr_root $IMAGE_ROOT \
        --output_dir experiments/eval_results_mrl \
        --batch_size 64 \
        --encoder_mode clip \
        --flickr_split val \
        --mrl_dims 512 256 128